In [48]:
# https://platform.olimpiada-ai.ro/en/problems/182

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
from sklearn.metrics import r2_score

In [49]:
train = pd.read_csv("/kaggle/input/datasets/abukanabek/voltline-dealership/train.csv")
test = pd.read_csv("/kaggle/input/datasets/abukanabek/voltline-dealership/test.csv")

train.shape, test.shape

((423, 26), (282, 25))

In [50]:
train.head()

,CarID,symboling,CarName,fueltype,aspiration,doornumber,carbody,drivewheel,enginelocation,wheelbase,...,enginesize,fuelsystem,boreratio,stroke,compressionratio,horsepower,peakrpm,citympg,highwaympg,price
0,400,0.0,Zenith Core Prime-400,gas,std,four,wagon,fwd,front,99.959654,...,179.0,mpfi,3.417887,3.266419,8.982752,153.0,5193,17.0,22.0,14579.0
1,224,0.0,Voltara Runner Z-224,gas,turbo,four,wagon,4wd,front,96.825986,...,107.0,mpfi,3.624777,2.632518,7.743210,116.0,4804,23.0,23.0,13007.0
2,405,1.0,Voltara Strider Sonic-405,gas,std,two,hatchback,fwd,front,99.088602,...,120.0,2bbl,3.391787,3.397099,8.899534,84.0,4759,26.0,32.0,9933.0
3,103,0.0,Titan Nova Z-103,gas,std,four,wagon,fwd,front,100.400000,...,181.0,mpfi,3.430000,3.270000,9.000000,152.0,5200,17.0,22.0,14399.0
4,109,0.0,Ion Flow Hyper-109,diesel,turbo,four,sedan,rwd,front,107.900000,...,152.0,idi,3.700000,3.520000,21.000000,95.0,4150,28.0,33.0,13200.0


In [51]:
pd.concat([train.isna().sum(), test.isna().sum()], axis=1)

,0,1
CarID,0,0.0
symboling,4,3.0
CarName,0,0.0
fueltype,0,0.0
aspiration,3,3.0
doornumber,1,1.0
carbody,3,3.0
drivewheel,0,1.0
enginelocation,2,0.0
wheelbase,0,0.0


In [52]:
from sklearn.model_selection import train_test_split

features = [c for c in train.columns if c not in ['CarID', 'price']]
cat_features = [c for c in train.select_dtypes('object').columns if c in features]
target_col = 'price'

X, y = train[features], train[target_col]
X_test = test[features]

cat_fill_values = pd.Series(train[cat_features].value_counts().idxmax(), index=cat_features)
X.loc[:, cat_features] = X.loc[:, cat_features].fillna(cat_fill_values)
X_test.loc[:, cat_features] = X_test.loc[:, cat_features].fillna(cat_fill_values)

X_train, X_valid, y_train, y_valid = train_test_split(X, y, test_size=0.1, random_state=42)
X_train.shape, X_valid.shape

((380, 24), (43, 24))

In [54]:
from catboost import Pool

train_pool = Pool(X_train, y_train, cat_features=cat_features)
valid_pool = Pool(X_valid, y_valid, cat_features=cat_features)

In [62]:
from catboost import CatBoostRegressor

params = {
    'iterations': 1000,
    'loss_function': 'MAE',
    'metric_period': 100,
    'eval_metric': 'R2',
    'random_state': 42,
    'max_depth': 4
}

model = CatBoostRegressor(**params)

model.fit(train_pool, eval_set=valid_pool)

0:	learn: -0.1306794	test: -0.1745602	best: -0.1745602 (0)	total: 3.26ms	remaining: 3.25s
100:	learn: 0.8599807	test: 0.8103310	best: 0.8103310 (100)	total: 200ms	remaining: 1.78s
200:	learn: 0.9586833	test: 0.9197308	best: 0.9197308 (200)	total: 395ms	remaining: 1.57s
300:	learn: 0.9683740	test: 0.9272659	best: 0.9272659 (300)	total: 593ms	remaining: 1.38s
400:	learn: 0.9738702	test: 0.9310415	best: 0.9310415 (400)	total: 793ms	remaining: 1.18s
500:	learn: 0.9760999	test: 0.9312452	best: 0.9312452 (500)	total: 987ms	remaining: 983ms
600:	learn: 0.9779530	test: 0.9323648	best: 0.9323648 (600)	total: 1.19s	remaining: 787ms
700:	learn: 0.9796987	test: 0.9325304	best: 0.9325304 (700)	total: 1.38s	remaining: 587ms
800:	learn: 0.9807937	test: 0.9324653	best: 0.9325304 (700)	total: 1.58s	remaining: 392ms
900:	learn: 0.9815312	test: 0.9327805	best: 0.9327805 (900)	total: 1.77s	remaining: 195ms
999:	learn: 0.9818549	test: 0.9326697	best: 0.9327805 (900)	total: 1.97s	remaining: 0us

bestTest = 

In [71]:
y_pred = model.predict(X_test)

subm = pd.DataFrame({
    'subtaskID': [1, 2] + [3] * len(test),
    'datapointID': [1, 1] + test['CarID'].tolist(),
    'answer': [train['enginetype'].value_counts().idxmax(), round(train['price'][train['fueltype'] == 'gas'].mean(), 2)] + y_pred.tolist()
})

subm.to_csv("submission.csv", index=False)

subm.head()

,subtaskID,datapointID,answer
0,1,1,ohc
1,2,1,12728.06
2,3,436,17060.875759
3,3,441,15700.343704
4,3,553,6136.42505
